In [1]:
import sagemaker
print(f"SageMaker Version: {sagemaker.__version__}")

/home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker_core/main/shapes.py:2509: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  class MonitoringDatasetFormat(Base):


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
SageMaker Version: 2.236.0


In [2]:
# %%
"""
Model Development for Hybrid Recommendation System
Local debugging before SageMaker training
Models: CF (NMF), CB (TF-IDF), NLP (SBERT)
"""

# Download and save SBERT model locally (run once)
from sentence_transformers import SentenceTransformer
# model = SentenceTransformer("all-MiniLM-L6-v2")
# model.save("src/sbert_model")
# print("SBERT model saved to src/sbert_model")

# %%
# Initialize SageMaker session and load data from S3
import sagemaker
import pandas as pd
from sagemaker import get_execution_role

role = get_execution_role()
sess = sagemaker.Session()
bucket = "my-ml-hybrid-recsys-bucket"
prefix = "recommender/input-data"

# Load datasets
ratings = pd.read_csv(f"s3://{bucket}/{prefix}/ratings.csv")
movies  = pd.read_csv(f"s3://{bucket}/{prefix}/movies.csv")
tags    = pd.read_csv(f"s3://{bucket}/{prefix}/tags.csv")

# %%
# Data preprocessing
tag_agg = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x.astype(str))).reset_index()
movies = movies.merge(tag_agg, on="movieId", how="left").fillna("")
movies["text"] = movies["title"] + " " + movies["genres"].str.replace("|", " ") + " " + movies["tag"]

# %%
# Train Collaborative Filtering with NMF (no surprise)
from sklearn.decomposition import NMF
import numpy as np

user_item_matrix = ratings.pivot_table(index="userId", columns="movieId", values="rating").fillna(0)
nmf = NMF(n_components=100, random_state=42, max_iter=500)
user_emb = nmf.fit_transform(user_item_matrix)
item_emb = nmf.components_

print("CF Model (NMF) training completed locally")

# %%
# Train Content-Based and NLP components
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF on genres
tfidf = TfidfVectorizer(stop_words="english")
genre_text = movies["genres"].str.replace("|", " ")
tfidf_mat = tfidf.fit_transform(genre_text)

# SBERT embeddings
nlp_model = SentenceTransformer("src/sbert_model")
nlp_emb = nlp_model.encode(movies["text"].tolist(), convert_to_numpy=True)

print("CB and NLP models ready for local testing")

/home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

CF Model (NMF) training completed locally


[03/04/26 06:09:56] INFO     Use pytorch device_name: cpu                                ]8;id=3234;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=62609;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py#189\189]8;;\

                    INFO     Load pretrained SentenceTransformer: src/sbert_model        ]8;id=382809;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=160210;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py#197\197]8;;\

                    WARNING  You try to use a model that was created with version       ]8;id=334770;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=597159;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py#1382\1382]8;;\
                             5.2.3, however, your version is 3.0.1. This might cause                               
                             unexpected behavior or errors. In that case, try to update                            
                             to the latest version.                                                                
                                                                                                                   
                                                                                                                   
                                                                                                                   

[03/04/26 06:09:57] INFO     2 prompts are loaded, with the keys: ['query', 'document']  ]8;id=903862;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=402418;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py#326\326]8;;\

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

CB and NLP models ready for local testing
